[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Wildertrek/catcher-in-the-cache/blob/main/notebooks/07_ipip_human_anchor.ipynb)

# 07: Same-Instrument Human Anchor (IPIP-HEXACO)

**Research question.** Does the canonical H<->A_HEX fusion exceed the *human*
level, and does it survive when the character is out-of-corpus -- measured on the
*identical* instrument humans took?

**Design.** Administer the exact 240-item IPIP-HEXACO inventory (the instrument
22,299 humans completed) to 3 frontier raters (Opus 4.6, GPT-4o, Gemini 3.1 Pro)
in persona self-report voice, for 20 canonical (in-cache) + 20 synthetic
(out-of-cache) characters; names redacted, items shuffled + neutrally labelled,
scored with the identical reverse-keying. Compare per-rater across-persona
r(H, A) to the human across-person r(H, A).

**Result (preview).** human **+0.45**; LLM-persona canonical **+0.449** (= human);
LLM-persona synthetic **-0.33** (recovers the designed decoupling). Delta = 0.78,
all 3 raters. The fusion tracks cache membership, not a fixed rater limit.
LLM side reproduces at $0 from cached ratings.

## Setup

**In Colab, run the cell below first.** It clones the companion repository and installs dependencies (~30 s), then changes into `notebooks/` so the relative paths in this notebook resolve. Run locally, the cell is a no-op.

In [1]:
# Colab setup: clone the companion repo + install dependencies (no-op when run locally).
import sys, os, subprocess
if "google.colab" in sys.modules:
    if not os.path.isdir("catcher-in-the-cache"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/Wildertrek/catcher-in-the-cache.git"], check=True)
    os.chdir("catcher-in-the-cache/notebooks")
    subprocess.run(["pip", "install", "-q", "-r", "../requirements.txt"], check=True)
    print("Colab setup complete. Working directory:", os.getcwd())

In [2]:
import json, statistics
from pathlib import Path
ART = Path("../paper_artifacts/pivot6_hexaco_atlas/ipip_persona")
if not ART.exists(): ART = Path("paper_artifacts/pivot6_hexaco_atlas/ipip_persona")
res = json.loads((ART / "ipip_persona_results.json").read_text())
hb = json.loads((ART / "human_baseline_result.json").read_text())  # cached, $0, no live fetch

print(f"{'rater':18s} {'canonical r(H,A)':>16s} {'synthetic r(H,A)':>16s}")
for rn, rv in res["raters"].items():
    c, s = rv["canonical"], rv["synthetic"]
    print(f"{rn:18s} {c['r_H_A']:>16} {s['r_H_A']:>16}   (n={c['n']}/{s['n']})")
agg = res["aggregate"]
print(f"\n{'MEAN':18s} {agg['canonical']['mean_r']:>16} {agg['synthetic']['mean_r']:>16}")
print(f"\nhuman IPIP between-person r(H,A) : {hb['r_H_A']:+.3f}  (n={hb['n_after_validity_filter']}, CI {hb['ci95']}; committed artifact)")
print(f"LLM canonical (in-cache)        : {agg['canonical']['mean_r']:+.3f}  -> ~human level (estimands differ: between-person vs across-character)")
print(f"LLM synthetic (out-of-cache)    : {agg['synthetic']['mean_r']:+.3f}  -> collapses + reverses")
print(f"Delta (canonical - synthetic)   : {agg['delta_canonical_minus_synthetic']:+.3f}  (PRIOR_DRIVEN if >= 0.15)")

rater              canonical r(H,A) synthetic r(H,A)
anthropic_opus46             0.6282          -0.3088   (n=20/20)
openai_gpt4o                 0.4131          -0.3839   (n=20/20)
google_gemini31              0.3045          -0.3045   (n=20/20)

MEAN                         0.4486          -0.3324

human IPIP between-person r(H,A) : +0.450  (n=22299, CI [0.4389, 0.4617]; committed artifact)
LLM canonical (in-cache)        : +0.449  -> ~human level (estimands differ: between-person vs across-character)
LLM synthetic (out-of-cache)    : -0.332  -> collapses + reverses
Delta (canonical - synthetic)   : +0.781  (PRIOR_DRIVEN if >= 0.15)


## The human anchor (optional, reproduces r=0.45 from the public data)

The human reference is recomputed from the Open-Source Psychometrics Project
IPIP-HEXACO sample (`openpsychometrics.org/_rawdata/HEXACO.zip`, n=22,299 after
validity filtering). The cell below fetches it and computes the human
across-person r(H, A) using the committed instrument keying
(`ipip_hexaco_instrument.json`). It needs network + pandas/numpy; the LLM result
above does not depend on it.

In [3]:
# Optional: reproduce the human r(H,A)=0.45 from the public IPIP-HEXACO sample
try:
    import io, zipfile, urllib.request, numpy as np, pandas as pd
    inst = json.loads((ART / "ipip_hexaco_instrument.json").read_text())["items"]
    z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(
        "http://openpsychometrics.org/_rawdata/HEXACO.zip", timeout=60).read()))
    df = pd.read_csv(z.open("HEXACO/data.csv"), sep="\t", low_memory=False)
    items = list(inst)
    for c in items: df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df[(df[items] >= 1).all(1) & (df[items] <= 7).all(1) & (df.V1 >= 4) & (df.V2 >= 4)]
    for c in items:
        if inst[c]["sign"] == -1: df[c] = 8 - df[c]
    H = df[[c for c in items if c[0] == "H"]].mean(1)
    A = df[[c for c in items if c[0] == "A"]].mean(1)
    print(f"human across-person r(H,A) = {np.corrcoef(H, A)[0,1]:+.3f}  (n={len(df)})")
except Exception as e:
    print("skipped (needs network/pandas):", str(e)[:80], "\nhuman r(H,A) = +0.45 (documented)")

human across-person r(H,A) = +0.450  (n=22299)


## Verdict

- **H-A2 (cache falsifier) confirmed on a human instrument:** canonical r(H,A)
  collapses from **+0.449** to **-0.33** out-of-corpus (Delta 0.78, all 3 raters).
- **Over-fusion refuted:** canonical (0.449) **matches** the human 0.45 in this
  same-instrument self-report (subject) mode; LLM raters are not less separable
  than humans on cached characters -- they reproduce the cached prior.
- This is **subject mode**; the judge-mode panel value (0.58-0.66) is higher and
  mode-specific. Both modes collapse out-of-cache, so the fusion tracks **cache
  membership**, not a fixed rater limit.
- Focused 3-rater control (n=20/group, wide intervals), no IRB; the gold test is
  the queued human HEXACO-PI-R panel on open text.